# Explore a Kalanos assessment
**CPU only · no API key · no dataset download**

1. Choose **Runtime → Run all**. Setup installs a small, pinned Markdown renderer and sanitizer; the example report then loads.
2. Coming from a Hub collection row? Click **Paste URL from clipboard** (the Hub copied that report's URL when you clicked Colab). Or paste a public report URL and click **Load assessment**.
3. Optionally click **Download snapshot** to save the original `audit.md` and retrieval metadata.

This viewer reads one existing public Markdown assessment, not the dataset or full HTML report. It does not run the grader. It preserves the export's findings, tables, coverage and limitations without inferring missing evidence. Episode IDs and raw measurements are not guaranteed by this export; follow the full-report link for further inspection. A score does not establish training success.


In [ ]:
# @title Setup: install rendering dependencies (collapsed)
%pip install -q markdown-it-py==3.0.0 bleach==6.2.0


In [ ]:
# @title Setup: settings, safe download and viewer helpers (collapsed)
import hashlib
import html
import http.client
import json
import re
import socket
import urllib.error
import urllib.request
import zipfile
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import unquote, urlsplit

from IPython.display import HTML, FileLink, display, clear_output
import ipywidgets as widgets
from markdown_it import MarkdownIt
import bleach

# --- Settings ---------------------------------------------------------------
HUB_NETLOCS = {"hub.kalanos.ai"}  # the only host this notebook will contact
LINK_NETLOCS = HUB_NETLOCS | {"huggingface.co"}  # hosts shown as clickable links
MAX_BYTES = 2 * 1024 * 1024  # refuse an audit.md larger than 2 MB
TIMEOUT_SECONDS = 20
USER_AGENT = "kalanos-colab-audit-viewer/1.0"
SOURCES = {"hf", "gh", "up"}  # report URL grammar: /d/[@tenant/]<source>/<org>/<name>
SEGMENT = re.compile(r"[A-Za-z0-9_.-]{1,100}")


class AuditError(Exception):
    """A problem the reader can act on. The message is shown as-is."""


@dataclass
class Report:
    """Where a report lives. Built from the URL alone, before anything is downloaded."""

    slug: str  # e.g. @kalanos/hf/lerobot/pusht
    name: str  # e.g. lerobot/pusht
    report_url: str
    audit_url: str


@dataclass
class Fetched:
    """The downloaded export, kept byte for byte, plus how and when we got it."""

    raw: bytes
    text: str
    url: str
    retrieved_at: str  # UTC, ISO 8601
    sha256: str


def resolve_report_url(text):
    """Turn a pasted Hub report link into a Report, or raise AuditError.

    Accepts the report page, its .md form, or its /audit.md form. Anything that is
    not https, not on the Hub host, or not shaped like a report path is refused.
    """
    try:
        parts = urlsplit((text or "").strip())
    except ValueError:
        raise AuditError("That URL is malformed. Paste a public Hub report link.") from None
    if parts.scheme != "https":
        raise AuditError("Paste an https:// link to a public Hub report.")
    if parts.netloc.lower() not in HUB_NETLOCS:
        raise AuditError(f"Only reports on {', '.join(sorted(HUB_NETLOCS))} are supported.")
    path = unquote(parts.path).rstrip("/")  # %40 -> @; query and fragment are ignored
    for suffix in ("/audit.md", ".md"):
        if path.endswith(suffix):
            path = path[: -len(suffix)]
            break
    segments = path.split("/")
    tenant = None
    if len(segments) == 6 and segments[2].startswith("@"):  # /d/@tenant/hf/org/name
        tenant, segments = segments[2][1:], segments[:2] + segments[3:]
    if len(segments) != 5 or segments[:2] != ["", "d"] or segments[2] not in SOURCES:
        raise AuditError("That is not a Hub report link. Expected /d/@owner/hf/<org>/<dataset>.")
    source, org, name = segments[2:]
    for part in [tenant, org, name] if tenant else [org, name]:
        if not SEGMENT.fullmatch(part) or set(part) <= {"."}:
            raise AuditError("The report path contains characters that are not allowed.")
    netloc = parts.netloc.lower()
    slug = f"{'@' + tenant + '/' if tenant else ''}{source}/{org}/{name}"
    base = f"https://{netloc}/d/{slug}"
    return Report(slug=slug, name=f"{org}/{name}", report_url=base, audit_url=base + "/audit.md")


class _SameHubRedirects(urllib.request.HTTPRedirectHandler):
    """Follow redirects only to https URLs on the approved Hub host."""

    def redirect_request(self, req, fp, code, msg, headers, newurl):
        target = urlsplit(newurl)
        if target.scheme != "https" or target.netloc.lower() not in HUB_NETLOCS:
            raise AuditError("The Hub redirected somewhere unexpected, so the download was stopped.")
        return super().redirect_request(req, fp, code, msg, headers, newurl)


def fetch_audit(audit_url, ssl_context=None):
    """Download one audit.md with a timeout and a size cap; return a Fetched.

    `ssl_context` exists only so tests can trust a local certificate.
    """
    # Validate even when this helper is called directly.
    audit_url = resolve_report_url(audit_url).audit_url
    request = urllib.request.Request(
        audit_url, headers={"User-Agent": USER_AGENT, "Accept": "text/markdown, text/plain;q=0.8"}
    )
    opener = urllib.request.build_opener(
        _SameHubRedirects, urllib.request.HTTPSHandler(context=ssl_context)
    )
    try:
        with opener.open(request, timeout=TIMEOUT_SECONDS) as response:
            kind = response.headers.get_content_type()
            if kind not in ("text/markdown", "text/plain"):
                raise AuditError(f"The Hub returned {kind}, not Markdown. Check the report link.")
            declared = response.headers.get("Content-Length", "")
            if declared.isdigit() and int(declared) > MAX_BYTES:
                raise AuditError(f"The export is larger than the {MAX_BYTES // 1024} KB limit.")
            raw = response.read(MAX_BYTES + 1)  # read one byte past the cap to detect overflow
            final_url = response.geturl()
            if declared.isdigit() and len(raw) < int(declared):
                raise AuditError("The download was cut short. Try again.")
    except urllib.error.HTTPError as error:
        if error.code == 404:
            raise AuditError("No public export was found at this address. Check the report link.") from None
        if error.code in (401, 403):
            raise AuditError("Access was denied. The report may require sign-in, or the service may be blocking this request.") from None
        if error.code == 429:
            wait = error.headers.get("Retry-After", "")
            retry = f"Try again in {wait} seconds." if wait.isdigit() else "Try again later."
            raise AuditError(f"The Hub is rate limiting requests. {retry}") from None
        raise AuditError(f"The Hub answered HTTP {error.code}. Try again later.") from None
    except (TimeoutError, socket.timeout):
        raise AuditError(f"The Hub did not answer within {TIMEOUT_SECONDS} seconds.") from None
    except urllib.error.URLError as error:
        if isinstance(error.reason, (TimeoutError, socket.timeout)):
            raise AuditError(f"The Hub did not answer within {TIMEOUT_SECONDS} seconds.") from None
        raise AuditError(f"Could not reach the Hub ({error.reason}). Check the runtime's internet access.") from None
    except (OSError, http.client.HTTPException) as error:
        raise AuditError(f"The connection to the Hub was interrupted ({type(error).__name__}). Try again.") from None
    if len(raw) > MAX_BYTES:
        raise AuditError(f"The export is larger than the {MAX_BYTES // 1024} KB limit.")
    try:
        text = raw.decode("utf-8")
    except UnicodeDecodeError:
        raise AuditError("The export is not valid UTF-8 text.") from None
    if not text.strip():
        raise AuditError("The export is empty.")
    return Fetched(
        raw=raw,
        text=text,
        url=final_url,
        retrieved_at=datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        sha256=hashlib.sha256(raw).hexdigest(),
    )

# A standard Markdown parser, with raw HTML and remote media disabled.
MARKDOWN = MarkdownIt("commonmark", {"html": False}).enable("table")
MARKDOWN.disable("image")
ALLOWED_TAGS = {"p", "br", "strong", "em", "s", "blockquote", "pre", "code",
                "h1", "h2", "h3", "h4", "h5", "h6", "ul", "ol", "li",
                "table", "thead", "tbody", "tr", "th", "td", "hr", "a"}


def safe_link(tag, name, value):
    if tag != "a" or name != "href":
        return False
    try:
        url = urlsplit(value)
        return url.scheme == "https" and url.netloc.lower() in LINK_NETLOCS
    except ValueError:
        return False


def render_markdown(text):
    """Keep tables, nesting and links; disallow HTML, scripts and media loads."""
    return bleach.clean(MARKDOWN.render(text), tags=ALLOWED_TAGS,
                        attributes=safe_link, protocols={"https"}, strip=True)


def read_identity(text):
    """Best-effort metadata only. Never use parsed fields to omit report text."""
    def field(label):
        match = re.search(r"(?im)^- " + re.escape(label) + r":\s*(.+)$", text)
        return match.group(1).strip().strip("`") if match else None
    return {"assessed": field("Assessed"),
            "source_revision": field("Source identity revision"),
            "report_id": field("Report ID")}


def snapshot_bytes(report, audit):
    """Build an archive in memory. Nothing is saved or downloaded automatically."""
    import io
    meta = {"report_url": report.report_url, "downloaded_from": audit.url,
            "retrieved_at_utc": audit.retrieved_at, "bytes": len(audit.raw),
            "sha256": audit.sha256, **read_identity(audit.text)}
    buffer = io.BytesIO()
    with zipfile.ZipFile(buffer, "w", zipfile.ZIP_DEFLATED) as archive:
        archive.writestr("audit.md", audit.raw)
        archive.writestr("snapshot.json", json.dumps(meta, indent=2) + "\n")
    return buffer.getvalue()


class ViewerState:
    """Invalidate the previous result before each request, including failed ones."""
    def __init__(self):
        self.report = None
        self.audit = None

    def clear(self):
        self.report = self.audit = None

    def load(self, url, fetcher=fetch_audit):
        self.clear()
        report = resolve_report_url(url)
        audit = fetcher(report.audit_url)
        self.report, self.audit = report, audit
        return report, audit


STYLE = """<style>
.kalanos {font:14px/1.55 system-ui,sans-serif;max-width:1000px;overflow-wrap:anywhere}
.kalanos table {border-collapse:collapse;display:block;overflow:auto;max-width:100%}
.kalanos th,.kalanos td {border:1px solid #8885;padding:7px 10px;text-align:left}
.kalanos h1{font-size:24px}.kalanos h2{font-size:19px;margin-top:24px}
.kalanos pre{overflow:auto}.kalanos .meta{font-size:12px;opacity:.8}
</style>"""


def show_audit(report, audit):
    links = (f'<a href="{html.escape(report.report_url, quote=True)}" '
             'target="_blank" rel="noopener noreferrer">Open full report ↗</a>')
    display(HTML(STYLE + '<div class="kalanos">' + links
                 + f'<p class="meta">Fetched {html.escape(audit.retrieved_at)} · '
                 + f'{len(audit.raw):,} bytes · SHA-256 {audit.sha256[:12]}…</p>'
                 + render_markdown(audit.text) + '</div>'))


def read_clipboard():
    """Text of the browser clipboard (Colab only). Raises AuditError with a plain next step if it cannot be read."""
    try:
        from google.colab import output
    except ImportError:
        raise AuditError("Clipboard paste works only in Colab. Paste the URL into the field instead.")
    try:
        text = output.eval_js("navigator.clipboard.readText()")
    except Exception:  # permission denied, no user gesture, unsupported browser
        raise AuditError("The browser blocked clipboard access. Click the field and press Ctrl/Cmd+V instead.")
    if not isinstance(text, str) or not text.strip():
        raise AuditError("The clipboard is empty. Copy a report URL from the Hub first.")
    return text.strip()[:2000]


def make_viewer(default_url):
    """One input, one load action, and an optional snapshot button."""
    state = ViewerState()
    url = widgets.Text(value=default_url, description="Report URL:",
                       layout=widgets.Layout(width="95%"))
    paste = widgets.Button(description="Paste URL from clipboard",
                           tooltip="Fills the field with the report URL you copied on the Hub, then loads it")
    load = widgets.Button(description="Load assessment", button_style="primary")
    save = widgets.Button(description="Download snapshot", disabled=True)
    output, download_output = widgets.Output(), widgets.Output()

    def invalidate(change=None):
        state.clear()
        save.disabled = True
        with output:
            clear_output(wait=False)
            print("Report changed. Click Load assessment to fetch it.")
        with download_output:
            clear_output(wait=False)

    def on_load(button=None):
        state.clear()
        save.disabled = True
        load.disabled = True
        with download_output:
            clear_output(wait=False)
        with output:
            clear_output(wait=False)
            print("Fetching the public Markdown summary…")
            try:
                report, audit = state.load(url.value)
                clear_output(wait=False)
                show_audit(report, audit)
                save.disabled = False
            except AuditError as error:
                clear_output(wait=False)
                print(f"Could not load assessment: {error}")
            except Exception as error:  # unexpected: report it, keep the viewer usable
                clear_output(wait=False)
                print(f"Could not load assessment: unexpected {type(error).__name__}: {error}")
            finally:
                load.disabled = False

    def on_paste(button=None):
        try:
            pasted = read_clipboard()
        except AuditError as error:
            with output:
                clear_output(wait=False)
                print(error)
            return
        url.value = pasted  # editing the field clears any previous assessment
        on_load()  # the URL is validated (https, Hub host only) inside state.load

    def on_save(button=None):
        with download_output:
            clear_output(wait=False)
            if state.audit is None:
                print("Load an assessment first.")
                return
            # Defensive identity check even if a widget change event was missed.
            try:
                current = resolve_report_url(url.value)
                if current.audit_url != state.report.audit_url:
                    raise AuditError("The report changed. Load it again before saving.")
            except AuditError as error:
                state.clear()
                save.disabled = True
                print(error)
                return
            name = re.sub(r"[^A-Za-z0-9_.-]+", "__", state.report.slug)
            stamp = state.audit.retrieved_at.replace("-", "").replace(":", "")
            path = Path(f"kalanos-audit-{name}-{stamp}.zip")
            try:
                path.write_bytes(snapshot_bytes(state.report, state.audit))
                print(f"Snapshot saved: {path.name}")
                try:
                    from google.colab import files
                except ImportError:
                    display(FileLink(str(path)))
                else:
                    files.download(str(path))
            except OSError as error:
                print(f"Could not save snapshot: {error}")

    url.observe(invalidate, names="value")
    paste.on_click(on_paste)
    load.on_click(on_load)
    save.on_click(on_save)
    display(widgets.VBox([url, widgets.HBox([paste, load, save]), output, download_output]))
    on_load()  # Run all loads the example once, but never downloads a snapshot.
    return state


## Choose a report and view results
The URL field below accepts a public report page or its `/audit.md` link. Editing it clears the previous assessment until you load the new one. Downloads are bounded to 2 MB with a 20-second socket timeout.


In [ ]:
# @title Assessment viewer
viewer = make_viewer("https://hub.kalanos.ai/d/@kalanos/hf/lerobot/utokyo_xarm_pick_and_place")


### About snapshots
Snapshots preserve the exact downloaded Markdown plus its URL, retrieval time and SHA-256. Parsed assessment identity is best-effort: missing fields remain null. A linked report may change after a new assessment. No snapshot is downloaded during **Run all**.

Setup requires access to PyPI. Report requests are restricted to `https://hub.kalanos.ai`; rendered report links are restricted to Hub and Hugging Face. Embedded media and raw HTML are not loaded. If access is denied, try the public report in your browser; this alone does not mean the report is private.
